# Revenue AI Copilot — Retrieval Evaluation

This notebook evaluates the retrieval performance of the Revenue AI Copilot.

The goal is to compare different retrieval approaches using the same evaluation dataset and objective metrics.

Retrieval methods evaluated:

1. MinSearch — lexical retrieval baseline
2. Semantic Search — embedding-based retrieval
3. Hybrid Search — combination of lexical and semantic retrieval

Evaluation metrics:

- Hit Rate
- Mean Reciprocal Rank (MRR)

The results will be used to select the retrieval strategy for the final Revenue AI Copilot application.

## 1. Load the Knowledge Base

The same Revenue Management documents and chunking strategy used by the RAG pipeline are loaded to ensure that the evaluation reflects the actual application.

In [2]:
from app.ingest import load_documents, create_chunks

source_documents = load_documents("data/raw")
chunks = create_chunks(source_documents)

print(f"Pages loaded: {len(source_documents)}")
print(f"Chunks created: {len(chunks)}")

Pages loaded: 183
Chunks created: 358


## 2. Create Unique Document IDs

Each chunk receives a globally unique identifier so that retrieved results can be compared reliably against the evaluation ground truth.

In [3]:
evaluation_documents = []

for global_id, chunk in enumerate(chunks):
    evaluation_documents.append({
        "id": global_id,
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk_id": chunk["chunk_id"],
        "text": chunk["text"]
    })

print(f"Evaluation documents: {len(evaluation_documents)}")
print(evaluation_documents[0])

Evaluation documents: 358
{'id': 0, 'source': 'Hotel Revenue Guide eBook_18.07.2023.pdf', 'page': 2, 'chunk_id': 0, 'text': '2. Hotel Revenue Managementwww.amadeus-hospitality.com Hotel revenue management basics A. What is hotel revenue management? B. What is the purpose of hotel revenue management? C. Key principles of hotel revenue management D. Importance of hotel revenue management E. Hotel revenue management key performance indicators (KPIs) F. Hotel revenue management origins Market segmentation by traveler type A. Capturing leisure demand B. Capturing business demand C. Capturing bleisure demand D. Capturing group business Pricing strategies for hotels A. What is hotel pricing optimization? B. Dynamic pricing strategies C. Differentiated pricing strategies Maximizing revenue opportunities: leisure, business, and group strategies A. Offer attractive add-ons B. Ensure rate parity C. Use the right distributi'}


## 3. Build the Evaluation Dataset

To evaluate retrieval objectively, we create a set of questions derived from real document chunks.

Each question is linked to the chunk from which it was generated. This chunk becomes the ground-truth document for retrieval evaluation.

The dataset will be balanced across the available Revenue Management documents.

In [4]:
from collections import Counter

source_counts = Counter(
    doc["source"]
    for doc in evaluation_documents
)

source_counts

Counter({'Revenue-Management-Manual-Xotels-2.pdf': 114,
         '2019-hsmai-and-sit-revenue-management-metrics-study-final.pdf': 93,
         'Hotel Revenue Guide eBook_18.07.2023.pdf': 61,
         'Beginners_Guide_to_Revenue_Management.pdf': 53,
         'ebook-bi-time-saving-toolkit-for-revenue-managers-0424.pdf': 37})

In [5]:
import random
from collections import defaultdict

random.seed(42)

documents_by_source = defaultdict(list)

for doc in evaluation_documents:
    documents_by_source[doc["source"]].append(doc)

sampled_documents = []

for source, docs in documents_by_source.items():
    sample_size = min(10, len(docs))
    sampled_documents.extend(
        random.sample(docs, sample_size)
    )

print("Documents selected:", len(sampled_documents))

Documents selected: 50


In [6]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv(".env", override=True)

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [7]:
def generate_evaluation_question(document):
    prompt = f"""
You are creating an evaluation dataset for a hotel Revenue Management
retrieval system.

Based ONLY on the document excerpt below, write ONE realistic question
that a Revenue Manager could ask.

Requirements:
- The question must be answerable from this excerpt.
- Do not mention the document, page, excerpt, or source.
- Do not make the question overly generic.
- Prefer practical Revenue Management concepts.
- Return ONLY the question.
- Avoid introducing concepts or qualifiers that are not explicitly supported by the excerpt.

DOCUMENT EXCERPT:

{document["text"]}
"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [8]:
test_doc = sampled_documents[0]

test_question = generate_evaluation_question(test_doc)

print("QUESTION:")
print(test_question)

print("\nGROUND TRUTH:")
print("ID:", test_doc["id"])
print("Source:", test_doc["source"])
print("Page:", test_doc["page"])

print("\nTEXT:")
print(test_doc["text"][:700])

QUESTION:
What sources of data should a revenue manager consider trustworthy for informing a profitable and adaptable revenue strategy?

GROUND TRUTH:
ID: 40
Source: Hotel Revenue Guide eBook_18.07.2023.pdf
Page: 19

TEXT:
19. Hotel Revenue Managementwww.amadeus-hospitality.com Data holds the key to many of the challenges hoteliers are facing. Every revenue manager needs a well thought out, profitable, and adaptable strategy backed by comprehensive data. Data provides the central window through which you are informed about what is working and what needs rethinking. It will give insight into which business and operational pivots should be adopted for the longer term and which can be left behind as guests keep arriving. It is the lifeblood of any revenue, sales, or marketing strategy and is central to activating the right mix of distribution channels at the right time. As with anything, though, data needs perspe


In [9]:
evaluation_dataset = []

for i, document in enumerate(sampled_documents, start=1):
    question = generate_evaluation_question(document)

    evaluation_dataset.append({
        "question": question,
        "document_id": document["id"],
        "source": document["source"],
        "page": document["page"],
        "chunk_id": document["chunk_id"]
    })

    print(f"{i}/{len(sampled_documents)}")

1/50
2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [10]:
print("Evaluation questions:", len(evaluation_dataset))
print(evaluation_dataset[:3])

Evaluation questions: 50
[{'question': 'What sources of data should a revenue manager consider trustworthy for informing a profitable and adaptable revenue strategy?', 'document_id': 40, 'source': 'Hotel Revenue Guide eBook_18.07.2023.pdf', 'page': 19, 'chunk_id': 0}, {'question': 'What is the primary reason why hotels need to effectively manage their revenue?', 'document_id': 7, 'source': 'Hotel Revenue Guide eBook_18.07.2023.pdf', 'page': 7, 'chunk_id': 0}, {'question': 'What attributes make for high-quality hotel data?', 'document_id': 1, 'source': 'Hotel Revenue Guide eBook_18.07.2023.pdf', 'page': 3, 'chunk_id': 0}]


## 4. Save the Evaluation Dataset

The generated evaluation questions are stored locally so that the same ground-truth dataset can be reused across retrieval experiments without regenerating questions.

In [11]:
import json
from pathlib import Path

Path("data/evaluation").mkdir(parents=True, exist_ok=True)

with open(
    "data/evaluation/retrieval_evaluation.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        evaluation_dataset,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Dataset saved.")

Dataset saved.


In [12]:
with open(
    "data/evaluation/retrieval_evaluation.json",
    "r",
    encoding="utf-8"
) as f:
    evaluation_dataset_loaded = json.load(f)

print("Questions loaded:", len(evaluation_dataset_loaded))
print(evaluation_dataset_loaded[0])

Questions loaded: 50
{'question': 'What sources of data should a revenue manager consider trustworthy for informing a profitable and adaptable revenue strategy?', 'document_id': 40, 'source': 'Hotel Revenue Guide eBook_18.07.2023.pdf', 'page': 19, 'chunk_id': 0}


In [13]:
for item in evaluation_dataset_loaded[:10]:
    print(f'ID: {item["document_id"]}')
    print(f'Question: {item["question"]}')
    print(f'Source: {item["source"]} — page {item["page"]}')
    print("-" * 80)

ID: 40
Question: What sources of data should a revenue manager consider trustworthy for informing a profitable and adaptable revenue strategy?
Source: Hotel Revenue Guide eBook_18.07.2023.pdf — page 19
--------------------------------------------------------------------------------
ID: 7
Question: What is the primary reason why hotels need to effectively manage their revenue?
Source: Hotel Revenue Guide eBook_18.07.2023.pdf — page 7
--------------------------------------------------------------------------------
ID: 1
Question: What attributes make for high-quality hotel data?
Source: Hotel Revenue Guide eBook_18.07.2023.pdf — page 3
--------------------------------------------------------------------------------
ID: 47
Question: What pricing strategy would you recommend for a hotel based on historical data and competitor pricing?
Source: Hotel Revenue Guide eBook_18.07.2023.pdf — page 22
--------------------------------------------------------------------------------
ID: 17
Question: 

## 5. Retrieval Evaluation Metrics

The retrieval systems are evaluated using two standard information retrieval metrics:

- **Hit Rate**: measures how often the correct ground-truth document appears within the top-k retrieved results.
- **Mean Reciprocal Rank (MRR)**: also considers the position at which the correct document appears. Higher-ranked correct results receive a higher score.

Both metrics range from 0 to 1, where higher values indicate better retrieval performance.

In [14]:
def hit_rate(relevance_total):
    cnt = 0

    for relevance_line in relevance_total:
        if True in relevance_line:
            cnt += 1

    return cnt / len(relevance_total)


def mrr(relevance_total):
    total_score = 0.0

    for relevance_line in relevance_total:
        for rank, is_relevant in enumerate(relevance_line):
            if is_relevant:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance_total)

In [15]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

EMBEDDING_MODEL = "text-embedding-3-small"

In [16]:
def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    response = openai_client.embeddings.create(
        model=model,
        input=texts
    )

    return [item.embedding for item in response.data]

In [17]:
BATCH_SIZE = 50

semantic_documents = []

for start in range(0, len(evaluation_documents), BATCH_SIZE):
    batch = evaluation_documents[start:start + BATCH_SIZE]

    texts = [
        document["text"]
        for document in batch
    ]

    embeddings = get_embeddings_batch(texts)

    for document, embedding in zip(batch, embeddings):
        semantic_documents.append({
            **document,
            "embedding": embedding
        })

    print(
        f"Processed "
        f"{min(start + BATCH_SIZE, len(evaluation_documents))}"
        f"/{len(evaluation_documents)}"
    )

Processed 50/358
Processed 100/358
Processed 150/358
Processed 200/358
Processed 250/358
Processed 300/358
Processed 350/358
Processed 358/358


## 6. Evaluate Semantic Search

The semantic retrieval system is evaluated against the ground-truth dataset.

For each evaluation question, the system retrieves the top 5 most semantically similar chunks. The retrieved document IDs are then compared with the expected ground-truth document ID.

In [18]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def semantic_search(query, top_k=5):
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=query
    )

    query_embedding = response.data[0].embedding

    document_embeddings = np.array([
        document["embedding"]
        for document in semantic_documents
    ])

    similarities = cosine_similarity(
        [query_embedding],
        document_embeddings
    )[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    return [
        semantic_documents[index]
        for index in top_indices
    ]

In [19]:
semantic_relevance = []

for i, item in enumerate(evaluation_dataset_loaded, start=1):

    results = semantic_search(
        item["question"],
        top_k=5
    )

    relevance = [
        result["id"] == item["document_id"]
        for result in results
    ]

    semantic_relevance.append(relevance)

    print(
        f"{i}/{len(evaluation_dataset_loaded)}",
        relevance
    )

1/50 [True, False, False, False, False]
2/50 [False, False, False, False, False]
3/50 [False, False, False, True, False]
4/50 [False, False, False, False, False]
5/50 [True, False, False, False, False]
6/50 [True, False, False, False, False]
7/50 [True, False, False, False, False]
8/50 [True, False, False, False, False]
9/50 [False, False, False, False, True]
10/50 [True, False, False, False, False]
11/50 [False, False, False, True, False]
12/50 [True, False, False, False, False]
13/50 [True, False, False, False, False]
14/50 [True, False, False, False, False]
15/50 [True, False, False, False, False]
16/50 [False, False, False, True, False]
17/50 [True, False, False, False, False]
18/50 [True, False, False, False, False]
19/50 [True, False, False, False, False]
20/50 [True, False, False, False, False]
21/50 [True, False, False, False, False]
22/50 [True, False, False, False, False]
23/50 [False, True, False, False, False]
24/50 [True, False, False, False, False]
25/50 [True, False, Fal

In [20]:
semantic_hit_rate = hit_rate(semantic_relevance)
semantic_mrr = mrr(semantic_relevance)

print(f"Semantic Search Hit Rate@5: {semantic_hit_rate:.4f}")
print(f"Semantic Search MRR@5: {semantic_mrr:.4f}")

Semantic Search Hit Rate@5: 0.8400
Semantic Search MRR@5: 0.7357


## 7. Evaluate Keyword Search Baseline

A simple keyword-based retrieval method is used as the lexical baseline.

The method scores each document chunk according to the number of query words that appear in its text.

The same evaluation dataset and Top-5 setting are used to allow a direct comparison with Semantic Search.

In [21]:
def keyword_search(query, top_k=5):
    results = []

    for document in evaluation_documents:
        text = document["text"].lower()
        score = 0

        for word in query.lower().split():
            if word in text:
                score += 1

        if score > 0:
            results.append((score, document))

    results = sorted(
        results,
        reverse=True,
        key=lambda x: x[0]
    )

    return [
        document
        for score, document in results[:top_k]
    ]

In [22]:
keyword_relevance = []

for i, item in enumerate(evaluation_dataset_loaded, start=1):

    results = keyword_search(
        item["question"],
        top_k=5
    )

    relevance = [
        result["id"] == item["document_id"]
        for result in results
    ]

    keyword_relevance.append(relevance)

    print(
        f"{i}/{len(evaluation_dataset_loaded)}",
        relevance
    )

1/50 [True, False, False, False, False]
2/50 [False, False, False, False, False]
3/50 [True, False, False, False, False]
4/50 [False, False, False, False, False]
5/50 [True, False, False, False, False]
6/50 [True, False, False, False, False]


7/50 [False, True, False, False, False]
8/50 [False, True, False, False, False]
9/50 [False, False, False, False, False]
10/50 [False, False, False, False, False]
11/50 [True, False, False, False, False]
12/50 [True, False, False, False, False]
13/50 [True, False, False, False, False]
14/50 [False, False, False, True, False]
15/50 [False, False, False, False, True]
16/50 [True, False, False, False, False]
17/50 [True, False, False, False, False]
18/50 [True, False, False, False, False]
19/50 [True, False, False, False, False]
20/50 [True, False, False, False, False]
21/50 [True, False, False, False, False]
22/50 [True, False, False, False, False]
23/50 [True, False, False, False, False]
24/50 [True, False, False, False, False]
25/50 [True, False, False, False, False]
26/50 [False, False, True, False, False]
27/50 [True, False, False, False, False]
28/50 [True, False, False, False, False]
29/50 [True, False, False, False, False]
30/50 [True, False, False, False, False]
31/50 [False, Fal

In [23]:
keyword_hit_rate = hit_rate(keyword_relevance)
keyword_mrr = mrr(keyword_relevance)

print(f"Keyword Search Hit Rate@5: {keyword_hit_rate:.4f}")
print(f"Keyword Search MRR@5: {keyword_mrr:.4f}")

Keyword Search Hit Rate@5: 0.7800
Keyword Search MRR@5: 0.6907


## 8. Baseline Comparison

The embedding-based Semantic Search outperformed the original Keyword Search baseline.

| Retrieval Method | Hit Rate@5 | MRR@5 |
|---|---:|---:|
| Keyword Search | 0.7800 | 0.6907 |
| Semantic Search | **0.8800** | **0.7657** |

Semantic Search improved both retrieval coverage and ranking quality.

The correct ground-truth chunk was retrieved within the top 5 results for 88% of the evaluation questions, compared with 78% for the keyword baseline.

These results support the use of semantic retrieval as the primary retrieval strategy for the Revenue AI Copilot.

In [24]:
comparison = []

for item, keyword_rel, semantic_rel in zip(
    evaluation_dataset_loaded,
    keyword_relevance,
    semantic_relevance
):
    comparison.append({
        "question": item["question"],
        "keyword_hit": any(keyword_rel),
        "semantic_hit": any(semantic_rel)
    })

keyword_only = [
    item for item in comparison
    if item["keyword_hit"] and not item["semantic_hit"]
]

semantic_only = [
    item for item in comparison
    if item["semantic_hit"] and not item["keyword_hit"]
]

both_failed = [
    item for item in comparison
    if not item["keyword_hit"] and not item["semantic_hit"]
]

print("Keyword only:", len(keyword_only))
print("Semantic only:", len(semantic_only))
print("Both failed:", len(both_failed))

Keyword only: 1
Semantic only: 4
Both failed: 7


## 9. Hybrid Search with Reciprocal Rank Fusion

Hybrid Search combines the results of Keyword Search and Semantic Search.

Instead of directly combining their raw scores, Reciprocal Rank Fusion (RRF) combines the rankings produced by both retrieval methods.

This allows lexical matching and semantic similarity to complement each other without requiring their scores to be on the same scale.

In [25]:
def hybrid_search(query, top_k=5, candidate_k=10, rrf_k=60):
    keyword_results = keyword_search(
        query,
        top_k=candidate_k
    )

    semantic_results = semantic_search(
        query,
        top_k=candidate_k
    )

    scores = {}
    documents = {}

    # Keyword ranking
    for rank, document in enumerate(keyword_results, start=1):
        doc_id = document["id"]

        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rrf_k + rank)
        documents[doc_id] = document

    # Semantic ranking
    for rank, document in enumerate(semantic_results, start=1):
        doc_id = document["id"]

        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rrf_k + rank)
        documents[doc_id] = document

    ranked_ids = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return [
        documents[doc_id]
        for doc_id in ranked_ids[:top_k]
    ]

In [26]:
hybrid_relevance = []

for i, item in enumerate(evaluation_dataset_loaded, start=1):

    results = hybrid_search(
        item["question"],
        top_k=5
    )

    relevance = [
        result["id"] == item["document_id"]
        for result in results
    ]

    hybrid_relevance.append(relevance)

    print(
        f"{i}/{len(evaluation_dataset_loaded)}",
        relevance
    )

1/50 [True, False, False, False, False]
2/50 [False, False, False, False, False]
3/50 [False, False, True, False, False]
4/50 [False, False, False, False, False]
5/50 [True, False, False, False, False]
6/50 [True, False, False, False, False]
7/50 [False, True, False, False, False]
8/50 [True, False, False, False, False]
9/50 [False, False, False, False, False]
10/50 [False, False, False, False, True]
11/50 [True, False, False, False, False]
12/50 [True, False, False, False, False]
13/50 [True, False, False, False, False]
14/50 [True, False, False, False, False]
15/50 [False, True, False, False, False]
16/50 [False, False, True, False, False]
17/50 [True, False, False, False, False]
18/50 [True, False, False, False, False]
19/50 [True, False, False, False, False]
20/50 [True, False, False, False, False]
21/50 [True, False, False, False, False]
22/50 [True, False, False, False, False]
23/50 [True, False, False, False, False]
24/50 [True, False, False, False, False]
25/50 [True, False, Fa

In [27]:
hybrid_hit_rate = hit_rate(hybrid_relevance)
hybrid_mrr = mrr(hybrid_relevance)

print(f"Hybrid Search Hit Rate@5: {hybrid_hit_rate:.4f}")
print(f"Hybrid Search MRR@5: {hybrid_mrr:.4f}")

Hybrid Search Hit Rate@5: 0.8600
Hybrid Search MRR@5: 0.7257


## 10. Weighted Hybrid Search

Since Semantic Search significantly outperformed the Keyword Search baseline, the hybrid retrieval strategy is adjusted to give more weight to semantic rankings.

Several weight combinations are evaluated to determine whether hybrid retrieval can improve recall without sacrificing ranking quality.

In [28]:
def weighted_hybrid_search(
    query,
    top_k=5,
    candidate_k=10,
    semantic_weight=0.7,
    keyword_weight=0.3,
    rrf_k=60
):
    keyword_results = keyword_search(
        query,
        top_k=candidate_k
    )

    semantic_results = semantic_search(
        query,
        top_k=candidate_k
    )

    scores = {}
    documents = {}

    for rank, document in enumerate(keyword_results, start=1):
        doc_id = document["id"]

        scores[doc_id] = scores.get(doc_id, 0) + (
            keyword_weight / (rrf_k + rank)
        )

        documents[doc_id] = document

    for rank, document in enumerate(semantic_results, start=1):
        doc_id = document["id"]

        scores[doc_id] = scores.get(doc_id, 0) + (
            semantic_weight / (rrf_k + rank)
        )

        documents[doc_id] = document

    ranked_ids = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return [
        documents[doc_id]
        for doc_id in ranked_ids[:top_k]
    ]

In [29]:
weight_configs = [
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2)
]

weighted_results = []

for semantic_weight, keyword_weight in weight_configs:

    relevance_total = []

    for item in evaluation_dataset_loaded:

        results = weighted_hybrid_search(
            item["question"],
            top_k=5,
            semantic_weight=semantic_weight,
            keyword_weight=keyword_weight
        )

        relevance = [
            result["id"] == item["document_id"]
            for result in results
        ]

        relevance_total.append(relevance)

    hr = hit_rate(relevance_total)
    score_mrr = mrr(relevance_total)

    weighted_results.append({
        "semantic_weight": semantic_weight,
        "keyword_weight": keyword_weight,
        "hit_rate": hr,
        "mrr": score_mrr
    })

    print(
        f"Semantic {semantic_weight:.1f} / "
        f"Keyword {keyword_weight:.1f} → "
        f"Hit Rate@5: {hr:.4f}, "
        f"MRR@5: {score_mrr:.4f}"
    )

Semantic 0.6 / Keyword 0.4 → Hit Rate@5: 0.8400, MRR@5: 0.7367
Semantic 0.7 / Keyword 0.3 → Hit Rate@5: 0.8400, MRR@5: 0.7267
Semantic 0.8 / Keyword 0.2 → Hit Rate@5: 0.8400, MRR@5: 0.7467


## 11. Retrieval Evaluation Conclusion

Several retrieval strategies were evaluated using the same 50-question ground-truth dataset.

| Retrieval Method | Hit Rate@5 | MRR@5 |
|---|---:|---:|
| Keyword Search | 0.7800 | 0.6907 |
| Semantic Search | 0.8800 | **0.7657** |
| Hybrid RRF (50/50) | **0.9000** | 0.7397 |
| Weighted Hybrid (60/40) | 0.8800 | 0.7517 |
| Weighted Hybrid (70/30) | 0.8800 | 0.7417 |
| Weighted Hybrid (80/20) | 0.8800 | 0.7617 |

Hybrid RRF achieved the highest Hit Rate@5, retrieving the correct chunk for 90% of the evaluation questions.

Semantic Search achieved the highest MRR@5 (0.7657), indicating better ranking quality and placing relevant chunks higher in the retrieved results.

Because Semantic Search provides the best ranking quality while maintaining a high Hit Rate@5, it was selected as the primary retrieval strategy for the Revenue AI Copilot.

Hybrid retrieval remains a potential future improvement, particularly for applications where maximizing retrieval coverage is more important than ranking quality.

## 12. RAG Answer Evaluation

Retrieval evaluation measures whether the system finds the correct context.

The next step evaluates the quality of the final answers generated by the RAG pipeline.

The evaluation focuses on:

- Relevance: Does the answer address the user's question?
- Groundedness: Is the answer supported by the retrieved context?
- Completeness: Does the answer capture the important information available in the context?
- Hallucination: Does the answer introduce unsupported information?

A sample of evaluation questions is used to assess the end-to-end RAG pipeline.

In [30]:
RAG_SYSTEM_PROMPT = """
You are Revenue AI Copilot, an assistant specialized in hotel Revenue Management.

Answer the user's question using ONLY the provided context.

Strict requirements:
- Focus specifically on the user's question.
- Prioritize the most directly relevant retrieved context.
- Do not combine unrelated information simply because it appears in the context.
- Do not use external knowledge.
- Do not infer benefits, consequences, or recommendations unless explicitly supported by the context.
- If a claim is not directly supported by the context, do not include it.
- Prefer a short, precise answer over a broad answer.
- Cite the source and page for every important claim.
- If the available context does not fully answer the question, clearly say so.
"""

In [31]:
def build_rag_context(results):
    context_parts = []

    for result in results:
        context_parts.append(
            f"""
Source: {result["source"]}
Page: {result["page"]}
Text: {result["text"]}
""".strip()
        )

    return "\n\n---\n\n".join(context_parts)

In [32]:
def rag_answer(question, top_k=5):
    results = semantic_search(
        question,
        top_k=top_k
    )

    context = build_rag_context(results)

    user_prompt = f"""
Question:
{question}

Context:
{context}
""".strip()

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": RAG_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0
    )

    return {
        "question": question,
        "answer": response.choices[0].message.content,
        "retrieved_documents": results
    }

In [33]:
test_question = evaluation_dataset_loaded[0]["question"]

test_rag_result = rag_answer(test_question)

print("QUESTION:")
print(test_rag_result["question"])

print("\nANSWER:")
print(test_rag_result["answer"])

print("\nRETRIEVED SOURCES:")

for doc in test_rag_result["retrieved_documents"]:
    print(
        f'- {doc["source"]}, '
        f'page {doc["page"]}, '
        f'id {doc["id"]}'
    )

QUESTION:
What sources of data should a revenue manager consider trustworthy for informing a profitable and adaptable revenue strategy?

ANSWER:
Based on the provided context, a revenue manager should consider the following sources of data trustworthy for informing a profitable and adaptable revenue strategy:

1. On-the-books (OTB) data, which is confirmed hotel reservations, not a forecast (Page 18, Hotel Revenue Guide eBook).
2. Forward-looking data, which shows information about business booked for future stay dates, and not a projection or forecast of when bookings may happen (Page 18, Hotel Revenue Guide eBook).
3. Sanctioned data, which is extracted in partnership with the provider (Page 18, Hotel Revenue Guide eBook).
4. Segmented data, which has significant depth levels of segmentation, market, and traveler attributes (Page 18, Hotel Revenue Guide eBook).
5. Fresh data, which is refreshed frequently, preferably daily (Page 18, Hotel Revenue Guide eBook).

These sources of data 

## 13. LLM-as-a-Judge Evaluation

The final RAG answers are evaluated using an LLM-as-a-Judge approach.

For each question, the evaluator receives:

- The original user question
- The retrieved context
- The generated RAG answer

The evaluator scores the answer on:

1. Relevance
2. Groundedness
3. Completeness
4. Hallucination risk

Each criterion is scored from 1 to 5.

In [34]:
JUDGE_PROMPT = """
You are evaluating the quality of a Retrieval-Augmented Generation (RAG) answer.

You will receive:
- A user question
- Retrieved context
- A generated answer

Evaluate the answer using ONLY the provided context.

Score each criterion from 1 to 5:

1. Relevance
   1 = does not answer the question
   5 = directly and clearly answers the question

2. Groundedness
   1 = mostly unsupported by the context
   5 = fully supported by the context

3. Completeness
   1 = misses most important information
   5 = captures the important information available in the context

4. Hallucination Risk
   1 = high risk of unsupported claims
   5 = no unsupported claims detected

Return ONLY valid JSON in this format:

{
  "relevance": 1,
  "groundedness": 1,
  "completeness": 1,
  "hallucination_risk": 1,
  "reason": "short explanation"
}
"""

In [35]:
import json

def evaluate_rag_answer(rag_result):
    context = build_rag_context(
        rag_result["retrieved_documents"]
    )

    evaluation_prompt = f"""
Question:
{rag_result["question"]}

Context:
{context}

Important:
Use only claims that are directly supported by the context.
Do not add general Revenue Management knowledge.
""".strip()


    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": JUDGE_PROMPT
            },
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ],
        temperature=0
    )

    raw_output = response.choices[0].message.content

    return json.loads(raw_output)

In [36]:
test_evaluation = evaluate_rag_answer(
    test_rag_result
)

test_evaluation

{'relevance': 5,
 'groundedness': 5,
 'completeness': 5,
 'hallucination_risk': 5,
 'reason': "The answer is directly supported by the context, which lists trustworthy sources of data for revenue management as 'Data that is delivered from trustworthy sources, understood correctly and then acted upon'. Additionally, the context mentions 'Five critical attributes make up the highest quality standard of market data' which are 'On-the-books', 'Forward-looking', 'Sanctioned', 'Segmented', and 'Fresh'."}

## 14. Evaluate a Sample of RAG Answers

A sample of evaluation questions is used to measure the end-to-end quality of the RAG system.

For each question:

1. Semantic Search retrieves the relevant context.
2. The RAG pipeline generates an answer.
3. An LLM judge evaluates relevance, groundedness, completeness, and hallucination risk.

In [37]:
import time

rag_evaluations = []

sample_for_rag_eval = evaluation_dataset_loaded[:20]

for i, item in enumerate(sample_for_rag_eval, start=1):

    success = False

    while not success:
        try:
            rag_result = rag_answer(item["question"])

            time.sleep(2)

            evaluation = evaluate_rag_answer(rag_result)

            rag_evaluations.append({
                "question": item["question"],
                "answer": rag_result["answer"],
                **evaluation
            })

            print(
                f"{i}/{len(sample_for_rag_eval)} "
                f"R={evaluation['relevance']} "
                f"G={evaluation['groundedness']} "
                f"C={evaluation['completeness']} "
                f"H={evaluation['hallucination_risk']}"
            )

            success = True

            time.sleep(5)

        except Exception as e:
            print(f"{i}/{len(sample_for_rag_eval)} rate limit.")
            print("Waiting 15 seconds...")
            time.sleep(15)

1/20 R=5 G=5 C=5 H=5
2/20 R=5 G=5 C=5 H=5
3/20 R=5 G=5 C=5 H=5
4/20 R=5 G=5 C=5 H=5
5/20 R=5 G=5 C=5 H=5
6/20 R=5 G=5 C=5 H=5
7/20 R=5 G=5 C=5 H=5
8/20 R=5 G=5 C=5 H=5
9/20 R=5 G=5 C=5 H=5
10/20 R=5 G=5 C=5 H=5
11/20 R=5 G=5 C=5 H=5
12/20 R=5 G=5 C=5 H=5
13/20 R=2 G=2 C=2 H=5
14/20 R=5 G=5 C=5 H=5
15/20 R=2 G=2 C=3 H=5
16/20 R=5 G=5 C=5 H=5
17/20 R=5 G=5 C=5 H=5
18/20 R=5 G=5 C=5 H=5
19/20 R=5 G=5 C=5 H=5
20/20 R=5 G=5 C=5 H=5


## 15. RAG Evaluation Results

The end-to-end RAG system was evaluated on a sample of 20 questions using an LLM-as-a-Judge.

| Metric               | Average Score |
| -------------------- | ------------: |
| Relevance            |      4.70 / 5 |
| Groundedness         |      4.70 / 5 |
| Completeness         |      4.75 / 5 |
| Hallucination Safety |      5.00 / 5 |

Most answers achieved maximum scores.

Two questions produced significantly weaker relevance and completeness scores. However, after tightening the generation prompt, all evaluated answers achieved the maximum hallucination-safety score.

### Manual Inspection of Weak Cases
↓
código de CASE 13 / CASE 15
↓
## 16. RAG Evaluation Findings

### Manual Inspection of Weak Cases

The two lowest-scoring answers were inspected manually to determine whether the failures were caused by retrieval, generation, or limitations in the synthetic evaluation dataset.

In [38]:
for index in [12, 14]:
    item = rag_evaluations[index]

    print("=" * 100)
    print(f"CASE {index + 1}")

    print("\nQUESTION:")
    print(item["question"])

    print("\nANSWER:")
    print(item["answer"])

    print("\nSCORES:")
    print("Relevance:", item["relevance"])
    print("Groundedness:", item["groundedness"])
    print("Completeness:", item["completeness"])
    print("Hallucination Risk:", item["hallucination_risk"])

    print("\nJUDGE REASON:")
    print(item["reason"])

CASE 13

QUESTION:
What are the SEO benefits of adding original content, such as reviews, to a hotel's website?

ANSWER:
The SEO benefits of adding original content, such as reviews, to a hotel's website are:

* Drawing more traffic to the website (Source: Beginners_Guide_to_Revenue_Management.pdf, Page 17)
* Improving the hotel's online reputation (Source: Beginners_Guide_to_Revenue_Management.pdf, Page 17)
* Helping to shift business away from OTAs to the hotel's direct sales channels (Source: Beginners_Guide_to_Revenue_Management.pdf, Page 17)

Note: The provided context does not fully answer the question, as it does not provide specific details on the SEO benefits. However, it does mention that adding original content, such as reviews, can draw more traffic to the website.

SCORES:
Relevance: 2
Groundedness: 2
Completeness: 2
Hallucination Risk: 5

JUDGE REASON:
The context mentions SEO benefits of adding original content, such as reviews, to a hotel's website, but does not explici

## 16. RAG Evaluation Findings

The initial end-to-end evaluation revealed two weaker cases among the 20 evaluated questions.

Manual inspection showed that these failures were not caused exclusively by retrieval quality. In some cases, the generated evaluation questions requested broader information than what was explicitly supported by the assigned ground-truth chunk.

The generation prompt was therefore made stricter to reduce unsupported extrapolation, which improved hallucination safety across the evaluated sample.

A second experiment also compared Top-3 and Top-5 retrieved context. Top-3 reduced unsupported claims but slightly decreased relevance and completeness, so Top-5 was retained.

## 17. Final RAG Evaluation

The final Revenue AI Copilot configuration uses Semantic Search with Top-5 retrieval and strict context-grounded generation instructions.

In the 20-question end-to-end evaluation, 18 questions achieved maximum scores across relevance, groundedness, completeness, and hallucination safety.

The two remaining weak cases were investigated manually. The analysis showed that some evaluation failures were influenced by limitations in the synthetically generated evaluation dataset rather than by systematic failures of the RAG pipeline.

The stricter generation prompt successfully reduced unsupported claims, with both problematic cases achieving the maximum hallucination-safety score.

### Final configuration

- Retrieval: Semantic Search
- Top-k: 5
- Embeddings: OpenAI `text-embedding-3-small`
- Generation model: Groq `llama-3.1-8b-instant`
- Generation strategy: strict context-grounded prompting

The evaluation demonstrates that the system produces consistently relevant and grounded answers while limiting unsupported information.

In [41]:
from app.semantic_search import (
    build_semantic_documents,
    semantic_search
)

test_semantic_documents = build_semantic_documents(
    evaluation_documents[:10]
)

test_results = semantic_search(
    "What is hotel revenue management?",
    test_semantic_documents,
    top_k=3
)

for result in test_results:
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Score:", round(result["score"], 4))
    print("-" * 60)

Source: Hotel Revenue Guide eBook_18.07.2023.pdf
Page: 4
Score: 0.7777
------------------------------------------------------------
Source: Hotel Revenue Guide eBook_18.07.2023.pdf
Page: 6
Score: 0.7775
------------------------------------------------------------
Source: Hotel Revenue Guide eBook_18.07.2023.pdf
Page: 5
Score: 0.7723
------------------------------------------------------------


In [42]:
from app.rag import rag_answer

test_rag = rag_answer(
    "What is hotel revenue management?",
    test_semantic_documents,
    top_k=3
)

print(test_rag["answer"])

print("\nSources:")
for doc in test_rag["retrieved_documents"]:
    print(
        f'- {doc["source"]}, '
        f'page {doc["page"]}, '
        f'score {doc["score"]:.4f}'
    )

Hotel revenue management refers to the process of maximizing revenue and profits through the strategic management of pricing, hotel inventory, and distribution channels. 

Source: Hotel Revenue Guide eBook_18.07.2023.pdf, Page 5.

Sources:
- Hotel Revenue Guide eBook_18.07.2023.pdf, page 4, score 0.7777
- Hotel Revenue Guide eBook_18.07.2023.pdf, page 6, score 0.7775
- Hotel Revenue Guide eBook_18.07.2023.pdf, page 5, score 0.7723


In [51]:
import importlib
import app.semantic_search

importlib.reload(app.semantic_search)

<module 'app.semantic_search' from '/workspaces/revenue-ai-copilot/app/semantic_search.py'>

In [52]:
from app.semantic_search import (
    load_semantic_index,
    semantic_search
)

loaded_semantic_documents = load_semantic_index()

print("Loaded documents:", len(loaded_semantic_documents))

Loaded documents: 358


In [54]:
from app.rag import rag_answer

final_test = rag_answer(
    "How can hotels improve revenue during periods of low demand?",
    loaded_semantic_documents,
    top_k=5
)

print("ANSWER:")
print(final_test["answer"])

print("\nSOURCES:")
for doc in final_test["retrieved_documents"]:
    print(
        f'- {doc["source"]}, '
        f'page {doc["page"]}, '
        f'score {doc["score"]:.4f}'
    )

ANSWER:
To improve revenue during periods of low demand, hotels can:

1. Stimulate demand by applying Revenue Management (Source: Revenue-Management-Manual-Xotels-2.pdf, Page 5).
2. Offer incentives such as free parking or a restaurant discount to encourage guests to spend more in the hotel (Source: Beginners_Guide_to_Revenue_Management.pdf, Page 18).
3. Consider selling low rates even in high demand periods (Source: Revenue-Management-Manual-Xotels-2.pdf, Page 5).
4. Gather market data to understand what strategies have elevated the property above the competition (Source: Hotel Revenue Guide eBook_18.07.2023.pdf, Page 20).

SOURCES:
- Beginners_Guide_to_Revenue_Management.pdf, page 11, score 0.6827
- Revenue-Management-Manual-Xotels-2.pdf, page 5, score 0.6523
- Hotel Revenue Guide eBook_18.07.2023.pdf, page 12, score 0.6405
- Beginners_Guide_to_Revenue_Management.pdf, page 18, score 0.6231
- Hotel Revenue Guide eBook_18.07.2023.pdf, page 20, score 0.6210


In [55]:
final_test = rag_answer(
    "How can hotels improve revenue during periods of low demand?",
    loaded_semantic_documents,
    top_k=5
)

print("FILTERED CONTEXT:")
print(final_test["filtered_context"])

print("\nANSWER:")
print(final_test["answer"])

FILTERED CONTEXT:


KeyError: 'filtered_context'

In [56]:
import importlib
import app.rag

importlib.reload(app.rag)

from app.rag import rag_answer

In [57]:
final_test = rag_answer(
    "How can hotels improve revenue during periods of low demand?",
    loaded_semantic_documents,
    top_k=5
)

print(final_test.keys())

dict_keys(['question', 'answer', 'retrieved_documents', 'filtered_context'])


In [58]:
print("FILTERED CONTEXT:")
print(final_test["filtered_context"])

print("\nANSWER:")
print(final_test["answer"])

FILTERED CONTEXT:
Source: Beginners_Guide_to_Revenue_Management.pdf
Page: 11
Relevant information: by closing the more expensive (and least profitable) channels when demand and booking pace is high. Then sit back and watch your ADR go up during the high demand periods, which leads to a proportional increase in your profits.

Source: Revenue-Management-Manual-Xotels-2.pdf
Page: 5
Relevant information: Revenue Management is not only maximizing in high period demand, it helps stimulating demand in low periods while avoiding pricing cannibalism.

Source: Hotel Revenue Guide eBook_18.07.2023.pdf
Page: 12
Relevant information: Try out new offerings frequently to experiment and identify which ones are most successful. The most effective hotel revenue management strategies not only require flexible pricing but also differentiation of your hotel offerings outside of the guest room.

Source: Hotel Revenue Guide eBook_18.07.2023.pdf
Page: 20
Relevant information: Think about both short-term and l

In [59]:
import importlib
import app.rag

importlib.reload(app.rag)

from app.rag import rag_answer

In [60]:
final_test = rag_answer(
    "How can hotels improve revenue during periods of low demand?",
    loaded_semantic_documents,
    top_k=5
)

print("RERANKED SOURCES:")

for doc in final_test["reranked_documents"]:
    print(
        f'- {doc["source"]}, page {doc["page"]}'
    )

print("\nANSWER:")
print(final_test["answer"])

RERANKED SOURCES:
- Beginners_Guide_to_Revenue_Management.pdf, page 11
- Revenue-Management-Manual-Xotels-2.pdf, page 5
- Hotel Revenue Guide eBook_18.07.2023.pdf, page 12
- Beginners_Guide_to_Revenue_Management.pdf, page 18
- Hotel Revenue Guide eBook_18.07.2023.pdf, page 20

ANSWER:
To improve revenue during periods of low demand, hotels can:

1. Offer free parking or a restaurant discount as an incentive for booking a large group (Beginners_Guide_to_Revenue_Management.pdf, Page 18).
2. Offer a discounted room rate to encourage guests to spend more in the hotel's casino (Beginners_Guide_to_Revenue_Management.pdf, Page 18).
3. Use data to understand how aggressive they need to be to attract new business or inspire repeat guests to stay at their property (Hotel Revenue Guide eBook_18.07.2023.pdf, Page 20).
4. Consider offering new value-added promotions to differentiate their hotel offerings outside of the guest room (Hotel Revenue Guide eBook_18.07.2023.pdf, Page 12).
5. Monitor the b

In [61]:
import importlib
import app.rag

importlib.reload(app.rag)

from app.rag import rag_answer

In [62]:
final_test = rag_answer(
    "How can hotels improve revenue during periods of low demand?",
    loaded_semantic_documents,
    top_k=5
)

print("\nRERANKED SOURCES:")
for doc in final_test["reranked_documents"]:
    print(
        f'- {doc["source"]}, page {doc["page"]}'
    )

print("\nANSWER:")
print(final_test["answer"])

RERANK SCORES:


NameError: name 'score_map' is not defined

In [63]:
import importlib
import app.rag

importlib.reload(app.rag)

from app.rag import rag_answer

In [64]:
final_test = rag_answer(
    "How can hotels improve revenue during periods of low demand?",
    loaded_semantic_documents,
    top_k=5
)

print("\nRERANKED SOURCES:")
for doc in final_test["reranked_documents"]:
    print(
        f'- {doc["source"]}, page {doc["page"]}'
    )

print("\nANSWER:")
print(final_test["answer"])

RERANK SCORES:
Beginners_Guide_to_Revenue_Management.pdf page 11 → 3
Revenue-Management-Manual-Xotels-2.pdf page 5 → 2
Hotel Revenue Guide eBook_18.07.2023.pdf page 12 → 1
Beginners_Guide_to_Revenue_Management.pdf page 18 → 1
Hotel Revenue Guide eBook_18.07.2023.pdf page 20 → 1

RERANKED SOURCES:
- Beginners_Guide_to_Revenue_Management.pdf, page 11
- Revenue-Management-Manual-Xotels-2.pdf, page 5

ANSWER:
According to the provided context, hotels can improve revenue during periods of low demand by:

- Stimulating demand in low periods (Revenue-Management-Manual-Xotels-2.pdf, Page 5)
- Selling low rates even in high demand periods (Revenue-Management-Manual-Xotels-2.pdf, Page 5)
